# M2 — Category

**Approach:** summary-only text classifier (contributor B), with a **group-aware split** and a **confidence analysis** (its original evaluation extra: mean confidence on correct vs incorrect predictions). Saves `models/m2_category.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
df = P.load_clean()
FEATS, TARGET = ["case_summary"], "category"
labels = sorted(df[TARGET].unique())
tr, te = P.group_split(df)
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
pred  = pipe.predict(te[FEATS])
proba = pipe.predict_proba(te[FEATS]); conf = proba.max(axis=1)
base = df[TARGET].value_counts(normalize=True).iloc[0]
metrics = P.evaluate(te[TARGET], pred, labels, baseline=base)
correct = (pred == te[TARGET].values)
metrics["confidence_gap"] = round(float(conf[correct].mean() - conf[~correct].mean()), 3)
metrics

In [ ]:
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, conf.round(3)
full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])
P.save_bundle("m2_category", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "multiclass",
    "label": "Category", "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"B", "approach":"summary-only text classifier",
             "split":"group-on-summary", "inputs":"summary only (no tags leakage)"}})